# 🌊 Real-Time Streaming & Async Agent Masterclass
### *Understanding Synchronous IO vs Async Generators, Token-by-Token Agent Streaming, and Real-Time STT/TTS Audio Streams*

---

## 🎯 What You Will Learn in this Notebook:
1. **What is Streaming?** — The fundamental difference between *Batch Processing* (waiting for everything) vs *Streaming* (processing chunks as they arrive).
2. **Synchronous vs Asynchronous Generators** — How Python's `yield`, `async def`, `async for`, and `asyncio.Queue` work behind the scenes.
3. **What is a "Stream" in a Voice Agent?** — Deep-dive into:
   - **STT Stream** (Raw 16kHz PCM Audio Chunks $\rightarrow$ Spoken Words)
   - **TTS Stream** (Text Token Chunks $\rightarrow$ Synthesized 44.1kHz WAV Audio Chunks)
4. **What is an Async Event Handler?** — How background threads, WebSocket callbacks, and async queues bridge data without blocking the main event loop.
5. **LangChain Agent Streaming with `astream()`** — How to stream token-by-token output and execute tools dynamically in real time.
6. **The Sandwich Streaming Pipeline** — Composing multiple async streams end-to-end like a Unix pipe (`STT | Agent | TTS`).

---

## 💡 Student Analogy: The Restaurant & The Water Hose

| Concept | Non-Streaming (Batch / Blocking) | Streaming (Chunked / Real-Time) |
| :--- | :--- | :--- |
| **Movie Analogy** | Downloading an entire 4GB movie file to your hard drive before you can press play. | Streaming Netflix: the first frame plays in 0.5 seconds while the rest buffers in the background. |
| **Restaurant Analogy** | The waiter waits until your appetizer, main course, dessert, and coffee are all cooked before bringing anything to your table. | The waiter brings the soup immediately, then the pasta, then the dessert as each is cooked. |
| **AI Voice Analogy** | Waiting 4 seconds for the LLM to write a 100-word paragraph before speaking a single syllable. | The AI speaks the first sentence within 300ms while still reasoning about the rest! |


---
# 📦 Section 1: What is Streaming in Python?

In standard Python programming, a regular function computes everything in memory and **returns** a single final value.
A **Generator** uses the keyword `yield` to return items **one piece at a time** as they become available.

Let's compare:
1. **Batch Function** (`return list`): Blocks until everything is done.
2. **Sync Generator** (`yield`): Gives items one at a time, but blocks execution between items.
3. **Async Generator** (`async def` + `yield` + `asyncio.sleep`): Streams items over time **without freezing** the application.


In [2]:
import time
import asyncio

# 1. BATCH FUNCTION (Non-streaming: Waits for the entire operation to finish)
def fetch_all_sentences_batch():
    print("⏳ [Batch] Fetching all sentences at once...")
    sentences = []
    for i in range(1, 4):
        time.sleep(1) # Simulates network delay
        sentences.append(f"Sentence {i}: The deli is now open!")
    return sentences

# Run Batch
start_time = time.time()
result = fetch_all_sentences_batch()
print(f"✅ [Batch Finished] Received {len(result)} sentences after {time.time() - start_time:.2f}s:")
for s in result:
    print("  ->", s)


⏳ [Batch] Fetching all sentences at once...
✅ [Batch Finished] Received 3 sentences after 3.01s:
  -> Sentence 1: The deli is now open!
  -> Sentence 2: The deli is now open!
  -> Sentence 3: The deli is now open!


Now let's see how an **Async Generator** transforms this into a real-time stream.
Instead of waiting 3 seconds to see anything, the caller gets the first item in 1 second, the second in 2 seconds, and the third in 3 seconds!


In [3]:
# 2. ASYNC GENERATOR (Streaming: Yields chunks as soon as they are ready)
async def fetch_sentences_stream():
    for i in range(1, 4):
        await asyncio.sleep(1) # Simulates waiting for AI / network chunk
        yield f"Sentence {i}: Welcome to The Daily Melt Deli!"

# Consuming an Async Stream using 'async for'
async def demo_async_stream():
    print("🌊 [Streaming] Starting to consume async stream...")
    start_time = time.time()
    
    async for sentence in fetch_sentences_stream():
        elapsed = time.time() - start_time
        print(f"  ⚡ [{elapsed:.2f}s] Received chunk: {sentence}")

# Run in event loop
await demo_async_stream()


🌊 [Streaming] Starting to consume async stream...
  ⚡ [1.00s] Received chunk: Sentence 1: Welcome to The Daily Melt Deli!
  ⚡ [2.00s] Received chunk: Sentence 2: Welcome to The Daily Melt Deli!
  ⚡ [3.00s] Received chunk: Sentence 3: Welcome to The Daily Melt Deli!


---
# 🎙️ Section 2: What is a "Stream" in our Voice Agent?

In our Voice Agent Sandwich Architecture, we deal with two types of streams:

```
[Browser Mic] ➔ 16kHz PCM Audio Stream ➔ [AssemblyAI STT Stream]
                                                    │
                                                    ▼
                                           Text Tokens Stream
                                                    │
                                                    ▼
                                         [LangChain Agent Stream]
                                                    │
                                                    ▼
                                           AI Response Stream
                                                    │
                                                    ▼
                                           [Cartesia TTS Stream] ➔ 44.1kHz WAV Audio Chunks ➔ [Browser Speaker]
```

### 1. The Speech-to-Text (STT) Stream
- **Input**: An infinite async stream of raw `bytes` (microphone audio arriving every 100ms).
- **Processing**: We send these bytes to AssemblyAI's WebSocket.
- **Output**: An async stream of transcription events:
  - `STTChunkEvent`: Partial words while the user is still talking.
  - `STTOutputEvent`: Finalized sentence triggered when Voice Activity Detection (VAD) detects silence ($>700$ms).

### 2. The Text-to-Speech (TTS) Stream & Sentence Boundary Chunking
Why can't we send single words to TTS?
- Single words lack intonation and sound robotic (e.g. "I... want... a... sandwich").
Why can't we wait for the entire paragraph?
- That adds 3-5 seconds of latency before the user hears anything!
- **The Solution**: **Sentence Boundary Buffering** — we accumulate tokens until we hit `.`, `!`, `?`, or `
`, synthesize that single sentence immediately, and yield the audio!


In [5]:
import re

# SIMULATED SENTENCE BOUNDARY TTS STREAM
async def simulated_tts_stream(token_stream):
    """
    Takes a stream of word tokens, buffers them into sentences,
    and yields synthesized audio chunks on sentence boundaries.
    """
    sentence_delimiter = re.compile(r'([.!?\n]+)')
    buffer = ""
    
    async for token in token_stream:
        buffer += token
        print(f"  [TTS Buffer Accumulating]: '{buffer}'")
        
        # Check if we have completed a sentence
        parts = sentence_delimiter.split(buffer)
        if len(parts) > 1:
            # Complete sentence formed!
            complete_sentence = parts[0] + parts[1]
            buffer = "".join(parts[2:]) # Remaining incomplete text
            
            # Simulate Cartesia synthesizing this sentence to audio bytes
            print(f"  🎙️ [TTS Synthesizing Sentence]: '{complete_sentence.strip()}' -> Generating Audio...")
            await asyncio.sleep(0.3) # Simulates Cartesia ~150ms latency
            audio_chunk = f"<Binary WAV Audio: '{complete_sentence.strip()}'>"
            yield audio_chunk
            
    # Flush any remaining text at the end
    if buffer.strip():
        print(f"  🎙️ [TTS Flushing Remaining]: '{buffer.strip()}'")
        yield f"<Binary WAV Audio: '{buffer.strip()}'>"

# Test stream of tokens coming from an AI model
async def simulated_llm_tokens():
    tokens = ["Hello", " there! ", "Welcome ", "to ", "The ", "Daily ", "Melt ", "Deli. ", "How ", "can ", "I ", "help ", "you ", "today?"]
    for t in tokens:
        await asyncio.sleep(0.15) # Simulates LLM token-by-token generation
        yield t

# Run the Sentence-Boundary TTS Stream
print("🚀 Starting Sentence-Boundary TTS Streaming Demo:")
async for audio_chunk in simulated_tts_stream(simulated_llm_tokens()):
    print(f"  🔊 PLAYING AUDIO CHUNK TO SPEAKER: {audio_chunk}\n")


🚀 Starting Sentence-Boundary TTS Streaming Demo:
  [TTS Buffer Accumulating]: 'Hello'
  [TTS Buffer Accumulating]: 'Hello there! '
  🎙️ [TTS Synthesizing Sentence]: 'Hello there!' -> Generating Audio...
  🔊 PLAYING AUDIO CHUNK TO SPEAKER: <Binary WAV Audio: 'Hello there!'>

  [TTS Buffer Accumulating]: ' Welcome '
  [TTS Buffer Accumulating]: ' Welcome to '
  [TTS Buffer Accumulating]: ' Welcome to The '
  [TTS Buffer Accumulating]: ' Welcome to The Daily '
  [TTS Buffer Accumulating]: ' Welcome to The Daily Melt '
  [TTS Buffer Accumulating]: ' Welcome to The Daily Melt Deli. '
  🎙️ [TTS Synthesizing Sentence]: 'Welcome to The Daily Melt Deli.' -> Generating Audio...
  🔊 PLAYING AUDIO CHUNK TO SPEAKER: <Binary WAV Audio: 'Welcome to The Daily Melt Deli.'>

  [TTS Buffer Accumulating]: ' How '
  [TTS Buffer Accumulating]: ' How can '
  [TTS Buffer Accumulating]: ' How can I '
  [TTS Buffer Accumulating]: ' How can I help '
  [TTS Buffer Accumulating]: ' How can I help you '
  [TTS Buff

---
# ⚡ Section 3: What is an Async Event Handler & Event Loop?

### The Challenge: Threaded Callbacks vs Asyncio Event Loops
Third-party SDKs (like AssemblyAI WebSocket client) run background C-threads or worker threads.
When a spoken word arrives, the SDK calls a synchronous callback function like `on_turn(client, event)` on its **background thread**.

If you try to call an `async` function or mutate async objects directly from a background thread, Python will crash with:
`RuntimeError: There is no current event loop in thread 'Thread-1'`

### The Solution: `asyncio.Queue` and `loop.call_soon_threadsafe()`
An **Async Event Handler** bridges the thread boundary safely by:
1. Capturing the main `loop = asyncio.get_running_loop()`.
2. Inside the background thread callback, using `loop.call_soon_threadsafe(queue.put_nowait, event)`.
3. The main async application reads from `await queue.get()` smoothly!


In [4]:
import threading

class AsyncThreadSafeBridgeDemo:
    def __init__(self):
        self.queue = asyncio.Queue()
        self.loop = asyncio.get_running_loop()
        
    # Synchronous callback triggered by an external SDK background thread
    def on_external_sdk_event(self, data: str):
        print(f"  [Background Thread: {threading.current_thread().name}] Received SDK event: '{data}'")
        # Safely transfer data into the asyncio queue on the main event loop
        self.loop.call_soon_threadsafe(self.queue.put_nowait, data)

    # Async generator consumer running on the main event loop
    async def event_stream(self):
        while True:
            item = await self.queue.get()
            if item == "STOP":
                break
            yield item

# Demonstration:
bridge = AsyncThreadSafeBridgeDemo()

# Simulate a background thread firing events (like AssemblyAI speech events)
def simulate_background_sdk():
    time.sleep(0.5)
    bridge.on_external_sdk_event("Speech Started")
    time.sleep(0.5)
    bridge.on_external_sdk_event("User said: 'I want a classic club'")
    time.sleep(0.5)
    bridge.on_external_sdk_event("Speech Ended (end_of_turn=True)")
    time.sleep(0.2)
    bridge.on_external_sdk_event("STOP")

# Start background thread
threading.Thread(target=simulate_background_sdk, name="AssemblyAI-Worker-Thread").start()

# Consume events on the main asyncio event loop
print("🎧 Listening to thread-safe async event stream:")
async for event in bridge.event_stream():
    print(f"  ✅ [Main Async Loop Received]: {event}")


🎧 Listening to thread-safe async event stream:
  [Background Thread: AssemblyAI-Worker-Thread] Received SDK event: 'Speech Started'
  ✅ [Main Async Loop Received]: Speech Started
  [Background Thread: AssemblyAI-Worker-Thread] Received SDK event: 'User said: 'I want a classic club''
  ✅ [Main Async Loop Received]: User said: 'I want a classic club'
  [Background Thread: AssemblyAI-Worker-Thread] Received SDK event: 'Speech Ended (end_of_turn=True)'
  ✅ [Main Async Loop Received]: Speech Ended (end_of_turn=True)
  [Background Thread: AssemblyAI-Worker-Thread] Received SDK event: 'STOP'


---
# 🧠 Section 4: LangChain Agent Streaming with `astream()`

### What is `astream()` in LangChain?
When building agents with LangChain, you have three execution methods:
1. `invoke()`: Blocking call. Waits for the entire agent reasoning, tool execution, and response synthesis before returning a single message.
2. `stream()`: Synchronous generator. Streams output steps.
3. `astream()`: **Asynchronous streaming generator**. Crucial for real-time voice applications because it allows concurrent tool calls and token-by-token streaming without blocking the server!

### Streaming Modes in LangChain:
- `stream_mode="values"`: Yields the full conversation state at each step.
- `stream_mode="updates"`: Yields only node updates in the graph.
- `stream_mode="messages"`: **Yields individual AIMessage token chunks as the LLM generates them** (this is what powers our voice agent!).


In [6]:
import os
from dotenv import load_dotenv

# Load API keys
load_dotenv(override=True)

# LangChain Imports
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 1. Define Sandwich Deli Domain Tools
MENU = {
    "classic club": 8.99,
    "grilled chicken panini": 9.50,
    "tuna melt": 8.50,
    "espresso": 3.00,
    "chips": 2.00
}

@tool
def get_menu() -> str:
    """Get the available deli sandwich menu and prices."""
    items = [f"{k.title()} (${v:.2f})" for k, v in MENU.items()]
    return "Our menu: " + ", ".join(items)

@tool
def order_sandwich(sandwich_name: str, quantity: int = 1) -> str:
    """Order sandwiches from the menu."""
    name = sandwich_name.lower()
    price = MENU.get(name, 8.50)
    total = price * quantity
    return f"Successfully added {quantity}x {name.title()} to order. Subtotal: ${total:.2f}."

tools = [get_menu, order_sandwich]

# 2. Initialize Model (SenseNova or fallback to OpenAI compatible)
sensenova_key = os.getenv("SENSENOVA_API_KEY")
if sensenova_key:
    llm = ChatOpenAI(
        base_url="https://token.sensenova.ai/v1",
        api_key=sensenova_key,
        model="sensenova-6.8-flash-lite",
        temperature=0.1
    )
    print("🧠 Connected to SenseNova LLM")
else:
    # Local simulated fallback if key is not present
    from langchain_core.language_models.fake import FakeListChatModel
    llm = FakeListChatModel(responses=["Welcome to The Daily Melt! Our Classic Club is $8.99."])
    print("⚠️ SENSENOVA_API_KEY not found; using mock model for demonstration.")

# 3. Create Agent with In-Memory Checkpointer
system_prompt = "You are Sam, a friendly, concise voice assistant for The Daily Melt Deli. Keep responses under 2 sentences."
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)
print("🥪 Voice Agent compiled successfully!")


🧠 Connected to SenseNova LLM
🥪 Voice Agent compiled successfully!


### Running `agent.astream(stream_mode="messages")` Token-by-Token
Let's see how tokens stream out in real time as the agent speaks!


In [8]:
import sys

async def test_agent_token_streaming(user_prompt: str):
    print(f"\n👤 User: '{user_prompt}'")
    print("🤖 Agent (Streaming tokens): ", end="", flush=True)
    
    config = {"configurable": {"thread_id": "session-1"}}
    input_payload = {"messages": [HumanMessage(content=user_prompt)]}
    
    # Stream using stream_mode='messages'
    async for chunk, meta in agent.astream(input_payload, config=config, stream_mode="messages"):
        # Check if the chunk is an AI message containing text tokens
        if isinstance(chunk, AIMessage) and chunk.content:
            # Print each token as it arrives without newline
            sys.stdout.write(chunk.content)
            sys.stdout.flush()
            await asyncio.sleep(0.02) # Visual pause to emphasize token arrival
            
    print("\n✨ [Turn Complete]")

# Test 1: Simple greeting
await test_agent_token_streaming("Hi! What sandwiches do you have?")

# Test 2: Tool calling (Ordering food)
await test_agent_token_streaming("I would like to order 2 classic clubs please!")



👤 User: 'Hi! What sandwiches do you have?'
🤖 Agent (Streaming tokens): 

We've got the Classic Club ($8.99), Grilled Chicken Panini ($9.50), and Tuna Melt ($8.50)! Would you like to order one?
✨ [Turn Complete]

👤 User: 'I would like to order 2 classic clubs please!'
🤖 Agent (Streaming tokens): 



We've got the Classic Club ($8.99), Grilled Chicken Panini ($9.50), and Tuna Melt ($8.50)! Would you like to order one?
✨ [Turn Complete]


---
# 🥪 Section 5: The End-to-End Streaming Sandwich Pipeline

Let's put everything together!
We will create a complete pipeline:
`Simulated STT Stream` $\rightarrow$ `LangChain Agent Stream` $\rightarrow$ `Sentence-Boundary TTS Stream` $\rightarrow$ `Speaker Playback`.

Watch how data flows through all 3 stages concurrently!


In [7]:
# Full End-to-End Pipeline Demonstration

async def simulated_stt_input():
    """Simulates voice recognition emitting completed user turns."""
    yield "Hi there, can you tell me what drinks you have?"
    await asyncio.sleep(2.0)
    yield "Please add 1 espresso to my order."

async def full_sandwich_voice_pipeline():
    print("=" * 70)
    print("🥪 RUNNING END-TO-END STREAMING SANDWICH PIPELINE")
    print("=" * 70)
    
    thread_id = "demo_live_thread"
    
    async for user_speech in simulated_stt_input():
        print(f"\n👂 [STT Detected User Speech]: '{user_speech}'")
        
        # Generator yielding tokens from the LangChain agent
        async def agent_token_generator():
            config = {"configurable": {"thread_id": thread_id}}
            input_data = {"messages": [HumanMessage(content=user_speech)]}
            async for chunk, meta in agent.astream(input_data, config=config, stream_mode="messages"):
                if isinstance(chunk, AIMessage) and chunk.content:
                    yield chunk.content
                    
        # Feed the agent token stream directly into the TTS sentence streamer!
        async for audio_chunk in simulated_tts_stream(agent_token_generator()):
            print(f"  🚀 [DISPATCHING TO WEBSOCKET -> SPEAKER]: {audio_chunk}")

# Execute full pipeline
await full_sandwich_voice_pipeline()


🥪 RUNNING END-TO-END STREAMING SANDWICH PIPELINE

👂 [STT Detected User Speech]: 'Hi there, can you tell me what drinks you have?'
  [TTS Buffer Accumulating]: '

I don't have information about drinks'
  🎙️ [TTS Synthesizing Sentence]: '' -> Generating Audio...
  🚀 [DISPATCHING TO WEBSOCKET -> SPEAKER]: <Binary WAV Audio: ''>
  [TTS Buffer Accumulating]: 'I don't have information about drinks on hand—'
  [TTS Buffer Accumulating]: 'I don't have information about drinks on hand—my menu tools are focused'
  [TTS Buffer Accumulating]: 'I don't have information about drinks on hand—my menu tools are focused on sandwiches. Would you like to see'
  🎙️ [TTS Synthesizing Sentence]: 'I don't have information about drinks on hand—my menu tools are focused on sandwiches.' -> Generating Audio...
  🚀 [DISPATCHING TO WEBSOCKET -> SPEAKER]: <Binary WAV Audio: 'I don't have information about drinks on hand—my menu tools are focused on sandwiches.'>
  [TTS Buffer Accumulating]: ' Would you like to see w

---
# 🎓 Key Takeaways for Students

| Feature | Regular Python (`def`) | Async Python (`async def`) |
| :--- | :--- | :--- |
| **Return Mechanism** | `return value` (Ends function immediately) | `yield item` (Yields chunk and pauses for next call) |
| **Iteration** | `for item in items:` | `async for item in stream:` |
| **Execution Style** | Blocking (one task at a time) | Non-blocking Concurrency (processes I/O while waiting) |
| **Agent Method** | `agent.invoke()` (Wait for full answer) | `agent.astream(stream_mode="messages")` (Token by token) |
| **Thread Bridge** | Shared variables with race conditions | `asyncio.Queue` + `loop.call_soon_threadsafe()` |

---
### 🥪 You are now ready to explain and build production-grade Voice AI Agents!
